In [1]:
import torch
import numpy as np

from dice import Dice, DiceFunctions
from functions import (
    MultiheadDiscreteSANetwork
)

In [2]:
DEVICE = 'cpu'
SEED = 784

In [3]:
states = torch.load('../../gym/cartpole/data/dqn_eps_02_inf/states.pt', weights_only=False)
actions = torch.load('../../gym/cartpole/data/dqn_eps_02_inf/actions.pt', weights_only=False)
rewards = torch.load('../../gym/cartpole/data/dqn_eps_02_inf/rewards.pt', weights_only=False)
dqn_actions = torch.load('../../gym/cartpole/data/dqn_eps_02_inf/dqn_actions.pt', weights_only=False)

In [4]:
state_dim = states[0].shape[1]
action_dim = 3
hidden_dim = 64
num_layers = 4

q_func = MultiheadDiscreteSANetwork(
    state_dim=state_dim,
    hidden_dim=hidden_dim,
    action_dim=action_dim,
    num_layers=num_layers,
    seed=SEED
)

w_func = MultiheadDiscreteSANetwork(
    state_dim=state_dim,
    hidden_dim=hidden_dim,
    action_dim=action_dim,
    num_layers=num_layers,
    seed=SEED
)

dice = Dice(
    q_function=q_func,
    w_function=w_func,
    gamma=0.99,
    q_lr=0.00001,
    w_lr=0.00001,
    lambda_lr=0.00001,
    f1_function=DiceFunctions.DUAL_DICE_P_3_2,
    f2_function=DiceFunctions.DUAL_DICE_P_3_2,
    method_name='best_dice',
    seed=SEED,
    device=DEVICE
)

In [5]:
dice.fit(
    state=states,
    action=actions,
    reward=rewards,
    target_action=dqn_actions,
    num_steps=150000,
    batch_size=2048,
    eval_iter=100,
    num_workers=8,
    result_folder='test_dqn'
)

  0%|          | 0/150000 [00:00<?, ?it/s]/Users/anthony/anaconda3/envs/dice/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
loss: -0.2213; value: 0.9405: 100%|██████████| 150000/150000 [15:38<00:00, 159.86it/s]


In [6]:
step_reward = dice.predict_per_step_reward(
    state=states, action=actions, reward=rewards
)

traj_reward = dice.predict_per_traj_reward(
    state=states, action=actions, reward=rewards
)

print(f"per step reward: {step_reward:.7f}")
print(f"per trajectory reward: {traj_reward:.7f}")

per step reward: 0.9404961
per trajectory reward: 235.1240367


In [7]:
print(f"last 300 avg: {np.mean(dice._experiment_data['value_per_step'][-300:])}")

last 300 avg: 0.9718588801225027


In [8]:
w = dice.predict_weights(np.concatenate(states), np.concatenate(actions))

w.min(),w.mean(), w.max()

(np.float32(7.194245e-12), np.float32(0.9591718), np.float32(5.9658427))